In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
import torch
import re
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

In [2]:
if torch.cuda.is_available():
    device = "cuda"
    print("✅ GPU kullanılıyor")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("⚠️ GPU yok, CPU kullanılıyor")

✅ GPU kullanılıyor
GPU: Tesla T4


In [3]:
df = pd.read_csv("articles_clean.csv")
df.head()

,Year,Title_TR,Abstract_TR,Keywords_TR,title_tr_clean,abstract_tr_clean,keywords_tr_clean,combined_text
0,2020-2021,İnsansız Hava Araçları için Manyetik Rezonans ...,İnsansız Hava Araçları (İHA) çeşitli alanlarda...,"insansız hava aracı (İHA), lityum batarya, man...",i̇nsansız hava araçları için manyetik rezonans...,i̇nsansız hava araçları (i̇ha) çeşitli alanlar...,"insansız hava aracı (i̇ha), lityum batarya, ma...",i̇nsansız hava araçları için manyetik rezonans...
1,2020-2021,İş Modeli Kanvas ve Yakın İş Modellerinin Savu...,Savunma ve havacılık sanayinin rekabetçi koşul...,"savunma ve uçak sanayii, kanvas iş modeli, üni...",i̇ş modeli kanvas ve yakın i̇ş modellerinin sa...,savunma ve havacılık sanayinin rekabetçi koşul...,"savunma ve uçak sanayii, kanvas iş modeli, üni...",i̇ş modeli kanvas ve yakın i̇ş modellerinin sa...
2,2020-2021,Hava Muharebesinde Otonom Savunma Algoritmasın...,"Bu çalışma kapsamında, temel hava muharebesi m...","bire-bir hava muharebesi, kural tabanlı yöntem...",hava muharebesinde otonom savunma algoritmasın...,"bu çalışma kapsamında, temel hava muharebesi m...","bire-bir hava muharebesi, kural tabanlı yöntem...",hava muharebesinde otonom savunma algoritmasın...
3,2020-2021,Modernization Projects,"Bu makalede, Türk Havacılık ve Uzay Sanayii LI...","artırılmış gerçeklik, model tabanlı izleme, iş...",modernization projects,"bu makalede, türk havacılık ve uzay sanayii li...","artırılmış gerçeklik, model tabanlı izleme, iş...","modernization projects bu makalede, türk havac..."
4,2020-2021,Döner Kanatlı Özgün Bir İHA Tasarımı ve Uçuş K...,Bu çalışmada 4 rotorlu döner kanatlı bir İnsan...,"İHA, motor, uçuş kontrol kartı, özgün yazılım.",döner kanatlı özgün bir i̇ha tasarımı ve uçuş ...,bu çalışmada 4 rotorlu döner kanatlı bir i̇nsa...,"i̇ha, motor, uçuş kontrol kartı, özgün yazılım.",döner kanatlı özgün bir i̇ha tasarımı ve uçuş ...


In [4]:
df["combined_text"].iloc[0]

'i̇nsansız hava araçları için manyetik rezonans kuplaj ile şarj i̇stasyonu tasarımı i̇nsansız hava araçları (i̇ha) çeşitli alanlarda kullanılabilen elektronik sistemlerdir. gerçekleştirilen bu çalışmada, i̇ha’da kullanılan lityum bataryalar için manyetik rezonans kuplaj yöntemi kullanılarak bir şarj istasyonu tasarlanmıştır. devre tasarımlarının analizleri, matlab ve pspice programları sayesinde gerçekleştirilmiştir. ayrıca; şarj cihazında kullanılan bobinler, ansys maxwell programı kullanılarak tasarlanmıştır. tasarlanan şarj istasyonu ile 12 cm’den 3w gücündeki lityum bataryalar şarj edilmiştir. tasarlanan sistem ile i̇ha’nın bataryası şarj olurken bile uçmasına imkân sağlanmaktadır. insansız hava aracı (i̇ha), lityum batarya, manyetik rezonanslı kuplaj, kablosuz güç transferi.'

In [5]:
df.isnull().sum()

,0
Year,0
Title_TR,0
Abstract_TR,0
Keywords_TR,0
title_tr_clean,0
abstract_tr_clean,0
keywords_tr_clean,0
combined_text,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 666 entries, 0 to 665
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Year               666 non-null    object
 1   Title_TR           666 non-null    object
 2   Abstract_TR        666 non-null    object
 3   Keywords_TR        666 non-null    object
 4   title_tr_clean     666 non-null    object
 5   abstract_tr_clean  666 non-null    object
 6   keywords_tr_clean  666 non-null    object
 7   combined_text      666 non-null    object
dtypes: object(8)
memory usage: 41.8+ KB


In [7]:
target = df["combined_text"]
target.head()

,combined_text
0,i̇nsansız hava araçları için manyetik rezonans...
1,i̇ş modeli kanvas ve yakın i̇ş modellerinin sa...
2,hava muharebesinde otonom savunma algoritmasın...
3,"modernization projects bu makalede, türk havac..."
4,döner kanatlı özgün bir i̇ha tasarımı ve uçuş ...


In [8]:
model_name = "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"
model = SentenceTransformer(model_name, device=device)

print(f"✅ Model loaded: {model_name}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded: emrecan/bert-base-turkish-cased-mean-nli-stsb-tr


In [9]:
# ağırlık katsayıları
W_TITLE = 0.20
W_ABSTRACT = 0.70
W_KEYWORDS = 0.10

print("Embedding üretimi başlıyor...\n")

# Hata almamak için null (boş) değerleri temizleyip string'e çeviriyoruz
titles = df["title_tr_clean"].fillna("").astype(str).tolist()
abstracts = df["abstract_tr_clean"].fillna("").astype(str).tolist()
keywords = df["keywords_tr_clean"].fillna("").astype(str).tolist()

# Bağımsız vektör üretimi
print("1/3: Başlıklar (Title) vektörleniyor...")
title_embs = model.encode(titles, convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=True) # L2 normalizasyonu biz yapacağımız için normalize_embeddings=False olmalı!

print("2/3: Özetler (Abstract) vektörleniyor...")
abs_embs = model.encode(abstracts, convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=True)

print("3/3: Anahtar Kelimeler (Keywords) vektörleniyor...")
key_embs = model.encode(keywords, convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=True)

# Ağırlıklandırma ve L2 Normalizasyonu
print("\nVektörler ağırlıklandırılıyor ve L2 Normalizasyonu uygulanıyor...")

# Matris çarpımı ile tek seferde ağırlıklı toplam (Formül: w1*E_title + w2*E_abs + w3*E_key)
weighted_embs = (W_TITLE * title_embs) + (W_ABSTRACT * abs_embs) + (W_KEYWORDS * key_embs)

# Satır bazında L2 Norm (Boy) hesaplama
norms = np.linalg.norm(weighted_embs, axis=1, keepdims=True)
norms[norms == 0] = 1.0

# Vektörü kendi boyuna bölerek normalize et (E_final)
final_embeddings_np = weighted_embs / norms

# Veri setine kaydetme
df["embedding"] = final_embeddings_np.tolist()

print("\nAğırlıklı Embedding işlemi başarıyla tamamlandı!")
print(f"Toplam kayıt: {len(df)}")
print(f"Yeni vektör boyutu: {len(df['embedding'].iloc[0])}")

Embedding üretimi başlıyor...

1/3: Başlıklar (Title) vektörleniyor...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

2/3: Özetler (Abstract) vektörleniyor...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

3/3: Anahtar Kelimeler (Keywords) vektörleniyor...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]


Vektörler ağırlıklandırılıyor ve L2 Normalizasyonu uygulanıyor...

Ağırlıklı Embedding işlemi başarıyla tamamlandı!
Toplam kayıt: 666
Yeni vektör boyutu: 768


In [10]:
# ağırlık katsayıları
W_TITLE = 0.20
W_ABSTRACT = 0.70
W_KEYWORDS = 0.10

# Metin Temizleme
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()                      # küçük harfe çevirir
    text = re.sub(r"\s+", " ", text)         # fazla boşlukları siler
    text = re.sub(r"[#%&*_=+<>]", "", text)  # belirli özel karakterleri siler
    return text.strip()

# Örnek Sorgu (Query)
raw_query_title = "Hava Muharebesinde Otonom Savunma Algoritmasının Geliştirilmesi"

raw_query_abstract = (
    "Bu çalışma kapsamında, temel hava muharebesi manevraları kullanılarak "
    "birebir muharebeler için otonom savunma algoritması geliştirilmiştir. "
    "Algoritma, hedef hava aracı ile beklenmedik bir şekilde karşılaşıldığı durumlarda "
    "saldırı üstünlüğünün sağlanması için en uygun muharebe manevrasını seçmeyi sağlamaktadır. "
    "Algoritmanın test edilmesi amacıyla saldıran ve savunan uçaklar için doğrusal olmayan "
    "dinamik modeller kullanılmıştır. Algoritmada manevra seçimi için temel savaş "
    "manevralarını içeren manevra kütüphanesinden uygun manevrayı seçecek kural tabanlı "
    "bir yapı önerilmiştir. MATLAB/Simulink ortamında yapılan benzetim çalışmaları ile "
    "algoritmanın başarımı test edilmiş ve sonuçlar gösterilmiştir."
)

raw_query_keywords = (
    "bire-bir hava muharebesi, kural tabanlı yöntem, temel hava muharebe manevraları"
)

# Bağımsız vektör üretimi
vec_title = model.encode(
    clean_text(raw_query_title),
    convert_to_numpy=True,
    normalize_embeddings=False
).astype(np.float32)

vec_abstract = model.encode(
    clean_text(raw_query_abstract),
    convert_to_numpy=True,
    normalize_embeddings=False
).astype(np.float32)

vec_keywords = model.encode(
    clean_text(raw_query_keywords),
    convert_to_numpy=True,
    normalize_embeddings=False
).astype(np.float32)

# Ağırlıklı birleştirme ve Normalizasyon -> Formül: w1 * E_title + w2 * E_abstract + w3 * E_keywords
weighted_query = (W_TITLE * vec_title) + (W_ABSTRACT * vec_abstract) + (W_KEYWORDS * vec_keywords)

# L2 Norm hesaplama
norm = np.linalg.norm(weighted_query)

# Vektörü kendi boyuna bölerek normalize etme
if norm > 0:
    final_query_vector = weighted_query / norm
else:
    final_query_vector = weighted_query

# Kosinüs Benzerliği Hesaplama
# Veri setimizdeki (daha önceden ağırlıklandırılmış) tüm embeddingleri matris olarak alıyoruz
corpus_embeddings = np.array(df["embedding"].tolist(), dtype=np.float32)

# Kosinüs benzerliğini hesaplama
similarity_scores = util.cos_sim(
    final_query_vector,
    corpus_embeddings
)[0]

# raporlama
df["similarity_score"] = similarity_scores.cpu().numpy()
top_results = df.sort_values(by="similarity_score", ascending=False)

print("--- En Benzer 5 Sonuç ---")
with pd.option_context('display.max_colwidth', None):
    display(top_results[["Year", "Title_TR", "Abstract_TR", "similarity_score"]].head(5))

print("\n")

print("--- En Alakasız 5 Sonuç ---")
with pd.option_context('display.max_colwidth', None):
    display(top_results[["Year", "Title_TR", "Abstract_TR", "similarity_score"]].tail(5))

--- En Benzer 5 Sonuç ---


,Year,Title_TR,Abstract_TR,similarity_score
2,2020-2021,Hava Muharebesinde Otonom Savunma Algoritmasının Geliştirilmesi,"Bu çalışma kapsamında, temel hava muharebesi manevraları kullanılarak birebir muharebeler için otonom savunma algoritması geliştirilmiştir. Algoritma, hedef hava aracı ile beklenmedik bir şekilde karşılaşıldığı durumlarda saldırı üstünlüğünün sağlanması için en uygun muharebe manevrasını seçmeyi sağlamaktadır. Algoritmanın test edilmesi amacıyla saldıran ve savunan uçaklar için doğrusal olmayan dinamik modeller kullanılmıştır. Algoritmada manevra seçimi için temel savaş manevralarını içeren manevra kütüphanesinden uygun manevrayı seçecek kural tabanlı bir yapı önerilmiştir. MATLAB/Simulink ortamında yapılan benzetim çalışmaları ile algoritmanın başarımı test edilmiş ve sonuçlar gösterilmiştir.",0.999934
479,2023-2024,"Hava Muharebesinde Oyun Teorisi, Bulanık Mantık ve Dinamik Programlama Kullanarak Karar Verme Algoritmasının Uygulanması ve Analizi","Bu çalışmada hava muharebesinde askeri pilotların başarı oranını artırmak için eğitim amaçlı kullanılmak üzere hava araçlarının aerodinamik yapıları göz önünde bulundurularak Unity oyun motorunda simülasyon ortamı hazırlanmıştır. Bu ortamda eğitim görmekte olan pilotlarla yarışacak, düşman pilot olarak görev alacak, pilotların manevralarını analiz ederek sürekli kendi stratejisini belirleyecek bir yapay zekâ modeli geliştirilmiştir. Yapay zekâ modelinin tasarımında derin öğrenme ve takviyeli öğrenme tekniklerini birleştiren Derin Q Ağları (DQN) kullanılmıştır. Bu alanda belirttiğimiz tekniklerle, geliştirdiğimiz eğitim simülasyon ortamının bir araya getirilmesi çalışmamızın temel katkısıdır. Geliştirilen simülasyon sistemi ve DQN tabanlı yapay zekâ modeli, hava muharebesi eğitimlerinde kullanılacak bir eğitim aracının temellerini oluşturmuştur.",0.825227
186,2021-2022,Otonom Hava Muharebelerinde Bulanık Metodlar Kullanılarak Karar Verme Algoritmalarının Gerçeklenmesi ve Analizi,"Hava muharebelerinde otonominin giderek yaygın- laşmasıyla birlikte görevleri yüksek başarı oranlarıyla gerçekleş- tirecek uçuş algoritmalarının önemi her geçen gün artmaktadır. Bu çalışmada ise bulanık mantık destekli karar verme algoritma- ları ile otonom hava muharebelerinde meydana gelebilecek du- rumlar için bulanık mantık kural tabanı ile MATLAB kullanıla- rak uçuş senaryolarının simülasyon ortamında gerçeklenmesi amaçlanmıştır. Uçakların it dalaşındaki etkinliği değerlendiril- mektedir. Algoritma, hedef uçak hareketine bağlı olarak bulanık mantık kural tabanına göre uygun referans değerlerini üreterek saldırı manevraları gerçekleştirmektedir.",0.824432
478,2023-2024,Hava Muharebesinde Bulanık Mantık ile Karar,"Hava muharebeleri, saldırı ve savunma manevralarının dengeli bir şekilde yapıldığı karmaşık stratejilere dayanmaktadır. Bu projede, hava muharebelerinde otonom karar alma yeteneğini artırmak için oyun teorisi, bulanık mantık ve dinamik programlama gibi ileri düzey tekniklerin birleşiminden oluşan bir algoritma geliştirilmesi hedeflenmektedir. Geliştirilecek olan algoritma, hava muharebelerinde bilinçli kararlar alabilen otonom sistemlerin geliştirilmesine odaklanmaktadır. Bu amaçla, geliştirilen algoritma simülasyon ortamında test edilecek ve başarı oranı ölçülecektir. Algoritmanın performansı, çeşitli simüle edilmiş muharebe senaryoları üzerinde test edilerek değerlendirilecektir. Başarı metrikleri arasında, misyon başarı oranları, kaynak kullanım verimliliği ve değişen ortamlara uygunluk gibi faktörler bulunacaktır. Bu çalışma, hava muharebelerinde otonom sistemlerin karar alma yeteneğini artırmak için kullanılabilecek yeni ve etkili bir yaklaşım sunmayı amaçlamaktadır.",0.813442
392,2023-2024,AGCAS Algoritması Tasarımı,"AGCAS Otomatik Çarpışma Algılama ve Önleme Sistemi Algoritması, özellikle jet eğitim uçaklarında öğrenci pilotların uçuşları sırasında uçuş güvenliğini arttırmak amacıyla geliştirilmiş bir sistemdir. Bu projede, gerçek dünya yükselti verilerine dayalı bir t



--- En Alakasız 5 Sonuç ---


,Year,Title_TR,Abstract_TR,similarity_score
602,2023-2024,Seramik Kesici Takımların Süper Alaşımlarda Kullanılmasının Malzemeye Olan Etkisinin Araştırılması,"Nikel bazlı süper alaşım grubundan olan Inconel 625, yüksek sıcaklık ve korozyon direncine rağmen havacılık ve enerji gibi sektörlerde en çok kullanılan malzemelerden biridir. Buna karşın işlenmesi zor olarak nitelendirilen bir malzemedir. Bu çalışmada, SiAlON seramik kesici takım kullanılarak, yüksek hızlarda frezeleme işlemi gerçekleştirilmiştir. Kuru kesme koşulları altında, gerçekleştirilen deneylerde kesme parametrelerinin yüzey pürüzlülüğüne ve yüzey altı tane yapısına etkisi deneysel olarak araştırılmıştır. Elde edilen sonuçlara göre, artan kesme hızına bağlı olarak tane yapısında belirgin bir yönelim gözlemlenmiştir. Artan kesme hızıyla birlikte deforme olmuş tabaka kalınlığı da artmaktadır. Kesme derinliği ve ilerleme hızının deforme olmuş tabaka kalınlığını etkilediği tespit edilmiştir.",0.239300
549,2023-2024,Manyetik Nanopartiküllerin Sentezi ve Elektromanyetik Kalkanlamaya Yönelik Geliştirilmesi,"Çalışma kapsamında karbon esaslı grafit, grafen oksit (GO), manyetik grafen oksit (MGO), kimyasal yolla indirgenmiş grafen oksit (rGO), kimyasal yolla indirgenmiş manyetik grafen oksit (rMGO), termal yolla indirgenmiş manyetik grafen oksit (tMGO), hem termal hem kimyasal yolla indirgenmiş manyetik grafen oksit (trMGO) ve grafite yüklenen manyetiklik özelliğiyle elde edilen grafen manyetik nanopartikül (Grafit MNP) malzemeleri sentezlenmiştir. Sentezlenen manyetik nanopartiküller (MNP), farklı yollarla indirgenmeden önce XRD, FT-IR, XPS ve SEM analizleri yapılmıştır. Ardından bahsi geçen her numuneye elektromanyetik girişim (EMI) testleri yapılmıştır. Yapılan testler sonucunda en umut verici malzemenin, çalışmanın ilk hedefi olan geçirgenlik<%5 değerine ulaşıldığı grafit olduğuna karar verilmiştir. MNP malzemesinin EMI deneylerindeki geçirgenlik özelliği üzerinde çok bir etkisi olmadığı gözlemlenmiş olsa da absorbans ve yansıma parametreleri üzerinde etkisi olabileceği düşünülmüştür.",0.219773
306,2022-2023,Karışık Sınır Şartlarının Araştırılması,"Bu belge, Plakların sınır şartlarının normalde kullanılan basit ve ankastre mesnet durumlarından farklı olarak incelenmesi ayrıca elastik sınır şartı ve ince bağlantı elemanı yöntemleri ile incelenmesi bunlar üstüne analizler yapılması ve perçin bağlantıların HyperSizer programına entegre edilmesi için kenar sabitlemesi oranı hesaplamalarını içermektedir.",0.215854
591,2023-2024,Resim Formatında Bulunan Grafikleri Görüntü İşleme Yöntemleriyle Dijital Veriye Dönüştürme,"Görsel grafikler, verileri anlaşılır ve etkili bir şekilde sunmanın yaygın bir yolu olarak kullanılmaktadır. Grafikler, web sayfaları, araştırma makaleleri veya sunum slaytları gibi dijital belgelerde veriyi sunmak için yaygın bir şekilde kullanılır. Karmaşık veriyi temsil etmek için etkili yöntemlerdir ve insanlara veriyi bütünsel olarak anlama konusunda yardımcı olurlar. Ancak görsel olarak verinin anlaşılması adına grafikler insan için ne kadar önemli olsa da altta yatan veriye bilgisayar tarafından doğrudan erişilmesi, analiz edilmesi ve dijital veri çıkarılması pek mümkün değildir. Makine okunabilirliğinin eksikliği görsel verinin üzerinde çalışılmasını ve yeniden kullanımını engeller. Savunma sanayide istihbarat, gözetleme ve keşif faaliyetlerinde kullanılan görüntü ve video gibi verilerin içinde bulunan grafiklerin makinelere anlamlı hale getirilmesi ve sistematik bir veri seti oluşturması açısından büyük fayda sağlaması beklenmektedir. Günümüzde, görüntü işleme teknikleri ve yapay zekâ algoritmaları, çeşitli uygulamalarda büyük bir öneme sahiptir. Bu teknikler, görüntülerin dönüştürülmesi ve analiz edilmesi için kullanılmaktadır. Bitirme tezi projesi ve Tusaş Lift Up projesi kapsamında, görsel (jpg, png vb.) formatındaki grafiklerin dönüşümünde, derin öğrenme algoritmaları yardımıyla grafiklerin yüksek doğruluk değerleriyle sınıflandırılması ve görüntü

In [11]:
df_finally = df[["Year","Title_TR", "Abstract_TR","Keywords_TR","combined_text","embedding"]]

In [12]:
with pd.option_context('display.max_colwidth', None):
    display(df_finally.head())

,Year,Title_TR,Abstract_TR,Keywords_TR,combined_text,embedding
0,2020-2021,İnsansız Hava Araçları için Manyetik Rezonans Kuplaj ile Şarj İstasyonu Tasarımı,"İnsansız Hava Araçları (İHA) çeşitli alanlarda kullanılabilen elektronik sistemlerdir. Gerçekleştirilen bu çalışmada, İHA’da kullanılan lityum bataryalar için manyetik rezonans kuplaj yöntemi kullanılarak bir şarj istasyonu tasarlanmıştır. Devre tasarımlarının analizleri, Matlab ve PSpice programları sayesinde gerçekleştirilmiştir. Ayrıca; şarj cihazında kullanılan bobinler, ANSYS Maxwell programı kullanılarak tasarlanmıştır. Tasarlanan şarj istasyonu ile 12 cm’den 3W gücündeki lityum bataryalar şarj edilmiştir. Tasarlanan sistem ile İHA’nın bataryası şarj olurken bile uçmasına imkân sağlanmaktadır.","insansız hava aracı (İHA), lityum batarya, manyetik rezonanslı kuplaj, kablosuz güç transferi.","i̇nsansız hava araçları için manyetik rezonans kuplaj ile şarj i̇stasyonu tasarımı i̇nsansız hava araçları (i̇ha) çeşitli alanlarda kullanılabilen elektronik sistemlerdir. gerçekleştirilen bu çalışmada, i̇ha’da kullanılan lityum bataryalar için manyetik rezonans kuplaj yöntemi kullanılarak bir şarj istasyonu tasarlanmıştır. devre tasarımlarının analizleri, matlab ve pspice programları sayesinde gerçekleştirilmiştir. ayrıca; şarj cihazında kullanılan bobinler, ansys maxwell programı kullanılarak tasarlanmıştır. tasarlanan şarj istasyonu ile 12 cm’den 3w gücündeki lityum bataryalar şarj edilmiştir. tasarlanan sistem ile i̇ha’nın bataryası şarj olurken bile uçmasına imkân sağlanmaktadır. insansız hava aracı (i̇ha), lityum batarya, manyetik rezonanslı kuplaj, kablosuz güç transferi.","[-0.017554597929120064, -0.022599617019295692, 0.007678552996367216, -0.02021501027047634, -0.024902313947677612, -0.015594638884067535, -0.07494257390499115, -0.0248128492385149, -0.04553448408842087, 0.07133074104785919, -0.0009209756972268224, 0.027788329869508743, 0.03445564582943916, -0.07433868199586868, 0.009737635031342506, 0.011438821442425251, 0.025486255064606667, -0.010009897872805595, -0.012079920619726181, 0.010503532364964485, 0.014989239163696766, -0.08899550139904022, -0.07513674348592758, -0.037781987339258194, 0.022418312728405, 0.0072859167121350765, -0.043061163276433945, 0.0021181663032621145, -0.04367204010486603, -0.039364349097013474, 0.038441311568021774, 0.04479195922613144, 0.0025378188583999872, 0.031242646276950836, -0.0759907066822052, -0.01921863667666912, -0.0025871815159916878, -0.0033367364667356014, 0.04382306709885597, -0.012762786820530891, -0.053430113941431046, -0.036138296127319336, 0.07277897000312805, -0.03297281265258789, 0.011281120590865612, 0.08432028442621231, 0.04130052775144577, -0.04949260503053665, 0.030667997896671295, 0.0029210420325398445, 0.061796754598617554, -0.032415810972452164, 0.05384518951177597, 0.03059462271630764, -0.007743783760815859, 0.0007015725132077932, 0.015234981663525105, -0.0046006301417946815, -0.021453741937875748, -0.023254601284861565, 0.00531542394310236, 0.00787939503788948, 0.000564116460736841, -0.04750950261950493, 0.0010164602426812053, 0.0345255471765995, 0.04870029538869858, -0.026386581361293793, 0.013370808213949203, 0.004512863699346781, -0.04606924206018448, 0.01630769856274128, 0.0056235999800264835, 0.06433004140853882, -0.01030718069523573, -0.037609875202178955, -0.05542679503560066, -0.04024723172187805, -0.011367881670594215, 0.03825536370277405, -0.04765075817704201, -0.04243004322052002, 0.05884571000933647, 0.009573354385793209, 0.017631929367780685, 0.013527154922485352, 0.00925667118281126, 0.007002171128988266, 0.03539164736866951, -0.010298884473741055, 0.018275074660778046, -0.03409991040825844, 0.019895045086741447, 0.010256542824208736, -0.07788006961345673, -0.04527560621500015, -0.008232757449150085, -0.015725580975413322, -0.00017155132081825286, 0.03999900072813034, ...]"
1,2020-2021,İş Modeli Kanvas ve Yakın İş Modellerinin Savunma ve Havacılık Sektörü Firmalarına Uygulanabilirliğinin İncelenm

In [13]:
# Kaydetme
df_finally.to_pickle("tusas_liftup_emrecan_bert_embeddings.pkl")

In [14]:
df_reloaded = pd.read_pickle("tusas_liftup_emrecan_bert_embeddings.pkl")
df_reloaded.head()

,Year,Title_TR,Abstract_TR,Keywords_TR,combined_text,embedding
0,2020-2021,İnsansız Hava Araçları için Manyetik Rezonans ...,İnsansız Hava Araçları (İHA) çeşitli alanlarda...,"insansız hava aracı (İHA), lityum batarya, man...",i̇nsansız hava araçları için manyetik rezonans...,"[-0.017554597929120064, -0.022599617019295692,..."
1,2020-2021,İş Modeli Kanvas ve Yakın İş Modellerinin Savu...,Savunma ve havacılık sanayinin rekabetçi koşul...,"savunma ve uçak sanayii, kanvas iş modeli, üni...",i̇ş modeli kanvas ve yakın i̇ş modellerinin sa...,"[0.02707720920443535, -0.010799337178468704, -..."
2,2020-2021,Hava Muharebesinde Otonom Savunma Algoritmasın...,"Bu çalışma kapsamında, temel hava muharebesi m...","bire-bir hava muharebesi, kural tabanlı yöntem...",hava muharebesinde otonom savunma algoritmasın...,"[0.013932212255895138, 0.012843888252973557, -..."
3,2020-2021,Modernization Projects,"Bu makalede, Türk Havacılık ve Uzay Sanayii LI...","artırılmış gerçeklik, model tabanlı izleme, iş...","modernization projects bu makalede, türk havac...","[-0.004494972061365843, -0.003270789748057723,..."
4,2020-2021,Döner Kanatlı Özgün Bir İHA Tasarımı ve Uçuş K...,Bu çalışmada 4 rotorlu döner kanatlı bir İnsan...,"İHA, motor, uçuş kontrol kartı, özgün yazılım.",döner kanatlı özgün bir i̇ha tasarımı ve uçuş ...,"[0.028957704082131386, -0.009388186037540436, ..."


In [15]:
df_reloaded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 666 entries, 0 to 665
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Year           666 non-null    object
 1   Title_TR       666 non-null    object
 2   Abstract_TR    666 non-null    object
 3   Keywords_TR    666 non-null    object
 4   combined_text  666 non-null    object
 5   embedding      666 non-null    object
dtypes: object(6)
memory usage: 31.3+ KB


In [16]:
# Görselleştirme
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# Veri Hazırlığı
embeddings = np.array(df['embedding'].tolist())

# t-SNE ve Kümeleme
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings)

kmeans = KMeans(n_clusters=6, random_state=42)
df['cluster'] = kmeans.fit_predict(embeddings)

# Grafik koordinatlarını ekle
df['x'] = embeddings_2d[:, 0]
df['y'] = embeddings_2d[:, 1]

# Plotly Grafiği Oluşturma
fig = px.scatter(
    df,
    x='x',
    y='y',
    color='cluster',
    hover_name='Title_TR',
    hover_data={
        'x': False,
        'y': False,
        'Year': True,
        'cluster': True
    },
    title='TUSAŞ LIFT UP Projeleri Anlamsal Benzerlik Uzayı (Etkileşimli t-SNE)',
    labels={'cluster': 'Küme No', 'Year': 'Yıl'},
    template='plotly_white',
    color_continuous_scale='Viridis'
)

# Grafik ayarlarını güncelle
fig.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    dragmode='pan',
    hoverlabel=dict(bgcolor="white", font_size=12, font_family="Arial")
)

fig.show()